## Initial Data Exploration

We start by importing the libraries needed for the analysis and inspecting the dataset to understand its structure before making any changes to the data.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns


In [ ]:
#Loading the dataset

data = pd.read_csv('../data/car_price_prediction.csv')

data.head(1)

## Duplicate Records

Duplicate observations can affect both the descriptive analysis and the models, so they are identified before moving to the preprocessing stage.


In [ ]:
#Exploring duplicates

data.duplicated().sum()

#There are 313 duplicates

In [ ]:
data["Levy"] = data["Levy"].replace("-", 0)
data["Levy"] = pd.to_numeric(data["Levy"])


data["Engine volume"] = data["Engine volume"].str.replace("Turbo","")
data["Engine volume"] = pd.to_numeric(data["Engine volume"])


data["Mileage"] = data["Mileage"].str.replace("km", '')
data["Mileage"]= pd.to_numeric(data["Mileage"])

#Transforming features to numeric so i can study them

## Dataset Overview

This section gives a first look at the size and structure of the dataset, including the number of observations and features.


In [ ]:
# #Exploring the dataset and information

print("Sample data :",data.sample(5))

print("Data information :",data.info())
print("Shape of the dataset :",data.shape)
print("Columns :",data.columns) 

data.isnull().sum()





#dataset contains no missing values , with datatypes of columns being either object or int64. , which is a continuous variable. The other columns are categorical variables, which will need to be encoded before training the model.








## Numerical Features

We examine the main statistics of the numerical variables to understand their ranges, central tendency, and variability.


In [ ]:
#Statistical summary & info
pd.set_option('display.float_format', '{:.2f}'.format)
display(data.describe())


#Price and mileage looks a bit awkward at max,std and mean, perhaps wrong values or considerable outliers.

display(data.describe(include="object"))


## Distribution and Skewness

The distributions of the numerical variables are examined to identify strong asymmetry and variables that may require special attention during preprocessing.


In [ ]:
#Analyzing every column and its distribution, outliers and relationship with the target variable.

for col in data.select_dtypes('number').columns:
    sns.kdeplot((data[col]))
    plt.show()
    sns.boxplot((data[col]))
    plt.show()

# #Price , Levy and Mileage seems very highly skewed, perhaps some massive outliers. We will need to handle them before training the model
# # we will try log transformation for now to take some ideas about normal distribution

for col in data.select_dtypes('number').columns:
    sns.kdeplot((np.log1p(data[col])))
    plt.show()
    sns.boxplot(np.log1p(data[col]))
    plt.show()


categories = data[["Leather interior","Fuel type","Gear box type","Drive wheels","Wheel"]]

for col in categories.columns:
    sns.countplot(x = data[col])
    plt.title(col)
    plt.show()

## Outlier Analysis

We inspect the numerical variables for unusually extreme values. This step is important because extreme observations can have a strong influence on a regression model.


In [ ]:
#Identifying outliers in suspected columns
ouliers_col = data[["Price","Levy","Mileage"]]

# for col in ouliers_col.columns:
#     sns.boxplot(data[col])
#     plt.show()
#     sns.boxplot(np.log1p(data[col]))
#     plt.show()

#Very massive ouliers detected
def detect_outliers(df, column):
    Q1 = df[column].quantile(0.25)
    Q3 = df[column].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 3 * IQR
    upper_bound = Q3 + 3 * IQR
    outliers = df[(df[column] <= lower_bound) | (df[column] >= upper_bound)]
    
    return outliers[column]

detect_outliers(data,"Price")
detect_outliers(data,"Levy")
detect_outliers(data,"Mileage")


# Mileage,Levy and Price contains statistical outliers according to the extreme IQR methode.


## Numerical Features

We examine the main statistics of the numerical variables to understand their ranges, central tendency, and variability.


In [ ]:
#Ctegorical analysis
display(data.describe(include="object"))

for col in data.select_dtypes('object').drop(columns=["Manufacturer","Model","Color","Category"]).columns:
    sns.countplot(x = data[col])
    plt.title(col)
    plt.show()


col2 = data[["Manufacturer","Model","Color","Category"]]   
for col in col2.columns:
    sns.countplot(y = data[col],order=data[col].value_counts().head(10).index)
    plt.title(col)
    plt.show()

#This part shows the distribution of categorical variables and then draws the countplot of every variable , i divided the code 





## Distribution and Skewness

The distributions of the numerical variables are examined to identify strong asymmetry and variables that may require special attention during preprocessing.


In [ ]:
for col in data.select_dtypes('number').columns:
    print(col)
    print("Skewness:", data[col].skew())
    print("Kurtosis:", data[col].kurt())
    print()


#Most numerical features show noticeable skewness and high kurtosis, indicating non-normal distributions and the presence of extreme observations. Price and Mileage show particularly high values, suggesting that outliers have a strong influence on their distributions. These observations will be considered during the preprocessing stage    

## Correlation Analysis

The correlation matrix helps us examine linear relationships between numerical variables and identify features that may be related to the target.


In [ ]:
#Studying correlation
numeric_data = data.select_dtypes('number')

numeric_data.corr()
sns.heatmap(numeric_data.corr(),annot=True)

#Most numerical features show weak linear correlations with Price. A strong correlation is observed between Engine volume and Cylinders, indicating that these two features are highly related



## Distribution and Skewness

The distributions of the numerical variables are examined to identify strong asymmetry and variables that may require special attention during preprocessing.


In [ ]:
#EDA Conclusions

# The dataset contains 19,237 observations and 18 features.
# No missing values were found in the dataset.
# The dataset contains both numerical and categorical features, with several categorical variables having a large number of unique values, particularly Model and Manufacturer.
# Price, Levy, and especially Mileage show strong skewness and extreme values, which may affect the model if they are not handled appropriately.
# The IQR analysis identified extreme statistical outliers in Price, Levy, and Mileage. These observations will be further evaluated and handled during the preprocessing stage.
# The categorical analysis shows that some categories are much more frequent than others. For high-cardinality features such as Manufacturer and Model, only the most frequent categories were visualized for better readability.
# The correlation analysis shows that most numerical features have weak linear correlations with Price.
# Engine volume and Cylinders show a relatively strong positive correlation (0.78), indicating that these two features are strongly related.
# Based on these findings, the next step is to clean and preprocess the data before building the prediction model